In [ ]:
%%writefile /content/requirements_rb.txt
pymupdf==2.0
requests==2.32.4
faiss-cpu==1.15.1

Overwriting /content/requirements_rb.txt


In [ ]:
# 설치 + 확인
!ls -al /content/requirements*
!pip install -r /content/requirements_rb.txt -q
!pip freeze | grep -E "pymupdf|requests|faiss"

-rw-r--r-- 1 root root 48 Nov  1 13:59 /content/requirements_rb.txt
faiss-cpu==1.15.1
pymupdf==2.0
pymupdf4llm==2.0
pymupdf_layout==2.0
requests==2.32.4
requests-oauthlib==2.0.0
requests-toolbelt==1.0.0


In [ ]:
# GPU 확인
import torch

print("GPU 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))

In [ ]:
# import
import json, time
import numpy as np
import requests

from google.colab import drive
drive.mount("/content/drive")

DATA = "/content/drive/MyDrive/KT 기본프로젝트"
PDF_PATH = f"{DATA}/SRD_CC_v5.2.pdf"
OUT_PATH = f"{DATA}/rules_new.jsonl"

OLLAMA_URL = "http://localhost:11434"
EMBED_MODEL = "bge-m3"
LLM_MODEL = "qwen2.5:7b"

In [ ]:
# Ollama 설치
!apt-get install -y -qq zstd
!curl -fsSL http://ollama.com/install.sh | sh
!ollama --version

In [ ]:
# 서버 실행 + 준비 확인
def wait_server(url, seconds=120):
    for _ in range(seconds):
        try:
            if requests.get(url, timeout=2).status_code == 200:
                print("서버 준비 완료:", url)
                return
        except Exception:
            pass
        time.sleep(1)
    print("서버가 아직 준비되지 않았습니다. 로그를 확인하세요")

!nohup ollama serve > ollama.log 2>&1 &
wait_server(OLLAMA_URL)

In [ ]:
# MODEL
!ollama pull bge-m3
!ollama pull qwen2.5:7b
!ollama list

In [ ]:
# 모델 동작 확인 (API 호출 세트)
res = requests.post(
    f"{OLLAMA_URL}/api/chat",
    json={
        "model": LLM_MODEL,
        "messages": [{"role": "user", "content": "1+1은?"}],
        "stream": False,        # 답을 한 번에 받겠다.
    },
    timeout=120,
)
res.raise_for_status()
print(res.json()["message"]["content"])

In [ ]:
# PDF열기 + 원본 확인
import pymupdf

doc = pymupdf.open(PDF_PATH)
print("전체 페이지 수:", len(doc))
print("메타데이터:", doc.metadata)

print(doc[0].get_text("text")[:300])    # 0번 = 첫 페이지

for page_no in range(10):
    if "Creative Commons" in doc[page_no].get_text("text"):
        print("CC 표기 발견: p.", page_no + 1)

In [ ]:
# 페이지 하나를 통쨰로 보기
PAGE_NO = 20
print(doc[PAGE_NO - 1].get_text("text")[:1500])

In [ ]:
# 글꼴 크기, 종류별 통계
from collections import Counter

stat = Counter()            # {(글꼴 크기, 글꼴 이름): 글자 수}
example = {}                # {(글꼴 크기, 글꼴 이름): 예시 문장}

for page in doc:
    data = page.get_text("dict")                # 글꼴 정보(dict로 확인)까지 포함
    for block in data["blocks"]:
        for line in block.get("lines", []):     # 이미지 블록에는 lines가 없어요
            for span in line["spans"]:          # span : 같은 글꼴로 이어진 글자 조각
                text = span["text"].strip()
                if not text:
                    continue
                key = (round(span["size"], 1), span["font"])
                stat[key] += len(text)
                example.setdefault(key, text[:40])      # setdefault : 키가 없을 때의 값

for key in sorted(stat, reverse=True):
    size, font = key
    print(f"{size:5} | {font:28} | 글자 수 {stat[key]:7,} | 예: {example[key]}")

In [ ]:
# 줄 단위로 읽는 helper + 제목 후보 목록
# 글꼴 정보와 글을 꺼내는 helper
def iter_lines(page):
    for block in page.get_text("dict")["blocks"]:
        for line in block.get("lines", []):
            spans = [s for s in line["spans"] if s["text"].strip()]
            if not spans:
                continue
            styles = {(round(s["size"], 1), s["font"]) for s in spans}  # 집합 컴프리헨션 {… for s in spans}: 리스트 컴프리헨션에 [ ] 대신 { }를 쓰면 중복이 사라짐
            text = "".join(s["text"] for s in spans).strip()
            yield styles, text, line["bbox"]        # yield: return처럼 값을 내놓되 함수가 끝나지 않고, for로 하나씩 꺼내 쓸 수 있는 방식

HEAD_CANDIDATES = [
    (26.0, "GillSans-SemiBold"),
    (18.0, "GillSans-SemiBold"),
    (14.8, "GillSans-SemiBold"),
    (14.0, "GillSans-SemiBold"),
    (12.0, "GillSans-SemiBold"),
    (10.5, "GillSans-SemiBold"),
]

found = {style: [] for style in HEAD_CANDIDATES}      # {글꼴: [(쪽, 글), ...]}

for page_no, page in enumerate(doc, start=1):
    for styles, text, bbox in iter_lines(page):
        if len(styles) != 1:                          # 글꼴이 섞인 줄은 제목이 아님
            continue        # len(styles) != 1이면 건너뜀: 한 줄에 글꼴이 섞이면 본문 속 강조 표현이지 제목이 아님
        style = list(styles)[0]
        if style in found:
            found[style].append((page_no, text))

for style in HEAD_CANDIDATES:
    items = found[style]
    print(f"\n{style} | 줄 수: {len(items)}")
    for page_no, text in items[:6] + items[-6:]:      # 앞 6개 + 뒤 6개
        print("   p.", page_no, "|", text)

In [ ]:
# 머리글, 쪽 번호 확인
HEADER_TEXT = "System Reference Document 5.2"

header_count = 0
pagenum_count = 0
for page in doc:
    for styles, text, bbox in iter_lines(page):
        if text == HEADER_TEXT:
            header_count += 1
        elif text.isdigit() and (11.0, "GillSans-SemiBold") in styles:
            pagenum_count += 1

print("머리글 줄 수:", header_count)          # 361이면 쪽마다 1개
print("쪽 번호 줄 수:", pagenum_count)        # 361이면 쪽마다 1개

In [ ]:
# Ex10. 정답 제목이 어느 크기에 있는지 보기

golden = json.load(open(f"{DATA}/golden.json", encoding="utf-8"))

# want는 문자열이거나 리스트라서, 전부 하나의 리스트로 통일
wants = []
for g in golden:
    w = g["want"]
    wants += [w] if isinstance(w, str) else w

for want in wants:
    exact = []                    # 글자가 완전히 같은 제목
    partial = []                  # 일부만 겹치는 제목
    for style in HEAD_CANDIDATES:
        for page_no, text in found[style]:
            if text == want:
                exact.append((style[0], page_no))
            elif want in text:
                partial.append((style[0], page_no, text))
    print(f"{want:34} | 정확히 일치: {exact[:3]} | 부분 일치 {len(partial)}건")
    if not exact and partial:
        print("      부분 예:", partial[:2])

In [ ]:
# Ex11. 제목 크기 → 단계, 청크 만들기
import re

FONT = "GillSans-SemiBold"
LEVEL = {                                   # (크기, 글꼴) → 단계
    (26.0, FONT): 1,                        # 장
    (18.0, FONT): 2,                        # 절 / 몬스터
    (14.0, FONT): 3,                        # 소절
    (12.0, FONT): 4,                        # 항목·용어·주문
    (14.8, FONT): 5,                        # 스탯블록 이름 (가장 깊은 단계)
}                                           # 10.5(표 제목)는 일부러 뺌 → 본문 취급
PAGE_NUM_STYLE = (11.0, FONT)
STAT = re.compile(r"\b(Str|Dex|Con|Int|Wis|Cha)\s*(\d{1,2})\s*([+\-−]\d+)\s*([+\-−]\d+)")

BODY_TITLE = "Playing the Game"             # 본문 첫 장 (그 앞 2~4쪽은 목차)
BODY_START = [p for p, t in found[(26.0, FONT)] if t == BODY_TITLE][0]   # Ex08 결과 재사용 → 5

def iter_lines_b(page):
    """iter_lines + 블록 번호 (같은 블록 = 같은 문단/제목)"""
    for b_no, block in enumerate(page.get_text("dict")["blocks"]):
        for line in block.get("lines", []):
            spans = [s for s in line["spans"] if s["text"].strip()]
            if not spans:
                continue
            styles = {(round(s["size"], 1), s["font"]) for s in spans}
            text = "".join(s["text"] for s in spans).strip()
            yield styles, text, b_no

def clean_join(lines):
    """줄 목록 → 한 문단. 탭 제거, 줄 끝 하이픈 이어붙이기"""
    out = ""
    for ln in lines:
        ln = ln.replace("\t", " ").strip()
        if not ln:
            continue
        if out.endswith("-") and ln[0].islower():   # proficien- + cies
            out = out[:-1] + ln
        else:
            out = out + " " + ln if out else ln
    return re.sub(r" {2,}", " ", out)

def fix_stat(body):
    """Str18+4 +4 → Str 18 (mod +4, save +4), MODSAVE 라벨 제거"""
    body = STAT.sub(r"\1 \2 (mod \3, save \4)", body)
    return re.sub(r"(?:MODSAVE\s*)+", "", body)

chunks = []        # 완성된 청크
dropped = []       # 본문이 없어 버린 제목
merged = []        # 두 줄 제목 합친 기록
unconv = []        # 스탯 변환 실패 기록
pre_lines = []     # 장 제목보다 먼저 읽힌 같은 쪽 본문 기록
path = {}          # {단계: 제목} 지금 위치

def open_chunk(title, level, page_no, b_no):
    for k in [k for k in path if k >= level]:
        del path[k]
    crumb = [path[k] for k in sorted(path) if k >= 1]
    path[level] = title
    return {"title": title, "level": level, "crumb": crumb, "block": b_no,
            "page": page_no, "page_end": page_no, "lines": []}

def close_chunk(c):
    raw = clean_join(c["lines"])
    body = fix_stat(raw)
    if not body:
        dropped.append((c["page"], c["title"]))
        return
    if "MODSAVE" in raw and "(mod " not in body:
        unconv.append((c["page"], c["title"]))
    chain = c["crumb"] + [c["title"]]
    chunks.append({
        "id": f"rule_{len(chunks) + 1:04d}",
        "text": " > ".join(chain) + "\n" + body,
        "meta": {
            "type": "meta" if chain[0] == "Legal Information" else "rule",   # 약관은 보존만
            "title": c["title"],
            "section": chain[1] if len(chain) > 1 else chain[0],
            "h1": chain[0],
            "page": c["page"],
            "page_end": c["page_end"],
            "source": "SRD_CC_v5.2",
        },
    })

cur = open_chunk("Front Matter", 0, 1, -1)

for page_no, page in enumerate(doc, start=1):
    if 1 < page_no < BODY_START:            # 목차 쪽(2~4)은 통째로 건너뜀
        continue
    if page_no == BODY_START:               # 박스 글이 장 제목보다 먼저 읽히므로 장을 미리 열어 둠
        close_chunk(cur)
        cur = open_chunk(BODY_TITLE, 1, page_no, -1)
    start_id, start_len = id(cur), len(cur["lines"])   # 이 쪽 시작 시점 기록

    for styles, text, b_no in iter_lines_b(page):
        style = list(styles)[0] if len(styles) == 1 else None

        # 1) 머리글·쪽 번호 / 2) 목차 점선 줄 버리기
        if text == HEADER_TEXT or (style == PAGE_NUM_STYLE and text.isdigit()):
            continue
        if re.search(r"\.{4,}", text):
            continue

        level = LEVEL.get(style)

        # 3) 제목이 아니면 본문
        if level is None:
            cur["lines"].append(text)
            cur["page_end"] = page_no
            continue

        # 4) 미리 열어 둔 장의 진짜 제목이 뒤늦게 나오면 무시
        if level == 1 and text == cur["title"]:
            continue

        # 5) 두 줄로 쪼개진 제목 합치기 (장 제목은 블록이 달라도 합침)
        if (level == cur["level"] and not cur["lines"] and cur["page"] == page_no
                and (level == 1 or b_no == cur["block"])):
            merged.append((cur["title"], text))
            cur["title"] += " " + text
            path[level] = cur["title"]
            continue

        # 6) 스탯블록 안에서 몬스터 이름이 한 번 더 찍힌 줄 건너뛰기
        if style == (14.8, FONT) and text == path.get(2):
            continue

        # 7) 새 장 제목: 같은 쪽에서 제목보다 먼저 읽힌 박스 글은 앞 청크에서 떼어 새 장으로 옮김
        carry = []
        if level == 1 and id(cur) == start_id and len(cur["lines"]) > start_len:
            carry = cur["lines"][start_len:]
            del cur["lines"][start_len:]
            cur["page_end"] = max(cur["page"], page_no - 1)
            pre_lines.append((page_no, text, len(carry)))
        close_chunk(cur)
        cur = open_chunk(text, level, page_no, b_no)
        cur["lines"] = carry

close_chunk(cur)
print("청크 수:", len(chunks), "| 본문 없어 버린 제목:", len(dropped), "| 두 줄 제목 합침:", len(merged))
print("스탯 변환된 청크:", sum("(mod " in c["text"] for c in chunks),
      "| MODSAVE 남은 청크:", sum("MODSAVE" in c["text"] for c in chunks))
print("스탯 변환 실패:", unconv)
print("장 제목 앞에 같은 쪽 본문이 먼저 붙은 경우:", pre_lines)
for a, b in merged[:15]:
    print("  합침:", a, "+", b)

# 변환 안 된 능력치가 남은 청크 수 (정상이면 0)
left = [c["id"] for c in chunks
        if re.search(r"\b(?:Str|Dex|Con|Int|Wis|Cha)\s*\d{1,2}\s*[+\-−]\d+\s*[+\-−]\d+", c["text"])]
print("미변환 능력치가 남은 청크:", len(left), left[:5])

청크 수: 1958 | 본문 없어 버린 제목: 69 | 두 줄 제목 합침: 9
스탯 변환된 청크: 335 | MODSAVE 남은 청크: 0
스탯 변환 실패: []
장 제목 앞에 같은 쪽 본문이 먼저 붙은 경우: [(89, 'Equipment', 5), (104, 'Spells', 4)]
  합침: Character + Creation
  합침: Barbarian Subclass: + Path of the Berserker
  합침: Bard Subclass: + College of Lore
  합침: Druid Subclass: + Circle of the Land
  합침: Monk Subclass: Warrior of the + Open Hand
  합침: Sorcerer Subclass: Draconic + Sorcery
  합침: Spell Scroll (Cantrip, 30 GP; + Level 1, 50 GP)
  합침: Curses and + Magical Contagions
  합침: Amulet of Proof against Detection + and Location
미변환 능력치가 남은 청크: 0 []


In [ ]:
# Ex11b. 빼기 전에 눈으로 확인
# (1) 머리글 문구가 남은 청크
for c in chunks:
    if HEADER_TEXT in c["text"]:
        i = c["text"].index(HEADER_TEXT)
        print("머리글 잔존:", c["id"], c["meta"]["h1"], "p.", c["meta"]["page"],
              "|", repr(c["text"][max(0, i - 60):i + 60]))

# (2) 뺄 후보 3개의 내용
for c in chunks:
    if c["meta"]["h1"] in {"Legal Information", "Contents", "Index of Stat Blocks"}:
        print("\n##", c["meta"]["h1"], "| 길이", len(c["text"]),
              "| p.", c["meta"]["page"], "-", c["meta"]["page_end"])
        print(c["text"][:400])

# (3) 본문이 없어 버려진 제목
print("\n버린 제목:", len(dropped))
print(", ".join(f"{p}:{t}" for p, t in dropped[:60]))

# (4) MODSAVE 위치 + 스탯블록 하나 전체
print("\nMODSAVE 포함 청크:", Counter(c["meta"]["h1"] for c in chunks if "MODSAVE" in c["text"]))
hag = [c for c in chunks if c["meta"]["title"] == "Green Hag"][0]
print("\n", hag["text"][:1500])

In [ ]:
# Ex11c. 1~5쪽에 뭐가 있는지
print(found[(26.0, FONT)])                    # 장 제목 26.0 줄이 어느 쪽에 있나
for p in (2, 3, 5):
    print(f"\n--- p.{p}")
    for styles, text, bbox in list(iter_lines(doc[p - 1]))[:16]:
        tag = "혼합" if len(styles) > 1 else str(list(styles)[0][0])
        print(f"{tag:>5} | {text[:70]}")

In [ ]:
# Ex11d. 장 제목 앞에 붙은 줄이 어느 쪽 내용인지 확인
for p in (89, 104):
    print(f"\n--- p.{p}")
    for styles, text, b_no in list(iter_lines_b(doc[p - 1]))[:16]:
        tag = "혼합" if len(styles) > 1 else str(list(styles)[0][0])
        print(f"{tag:>5} | b{b_no:<2} | {text[:70]}")


--- p.89
 11.0 | b0  | System Reference Document 5.2
 11.0 | b0  | 89
   혼합 | b2  | Selling Equipment
  9.0 | b3  | Equipment fetches half its cost when sold. In con-
  9.0 | b3  | trast, trade goods and valuables—like gems and art
  9.0 | b3  | objects—retain their full value in the marketplace.
  9.0 | b3  | “Magic Items” has prices for magic items.
 26.0 | b4  | Equipment
 18.0 | b5  | Coins
 10.0 | b5  | Characters often find coins on their adventures
 10.0 | b5  | and can spend those coins in shops, inns, and other
 10.0 | b5  | businesses. Coins come in different denominations
 10.0 | b5  | based on the relative worth of their material. The
 10.0 | b5  | Coin Values table lists coins and how much they’re
 10.0 | b5  | worth relative to the Gold Piece, which is the game’s
 10.0 | b5  | main coin. For example, 100 Copper Pieces are

--- p.104
 11.0 | b0  | System Reference Document 5.2
 11.0 | b0  | 104
   혼합 | b2  | Castingin Armor
  9.0 | b3  | You must have training with any ar

In [ ]:
# Ex12. 숫자로 확인
import random

lens = sorted(len(c["text"]) for c in chunks)
print("길이 min/중앙/max:", lens[0], lens[len(lens) // 2], lens[-1])
print("빈 텍스트:", sum(not c["text"].strip() for c in chunks))
print("탭 잔존:", sum("\t" in c["text"] for c in chunks))
print("머리글 잔존:", sum(HEADER_TEXT in c["text"] for c in chunks))
print("점선 잔존:", sum(bool(re.search(r"\.{4,}", c["text"])) for c in chunks))

covered = set()
for c in chunks:
    covered.update(range(c["meta"]["page"], c["meta"]["page_end"] + 1))
missing = [p for p in range(1, doc.page_count + 1) if p not in covered]
print("청크가 닿지 않은 쪽:", len(missing), missing[:10])

print("\n[장별 청크 수]")
for h1, n in Counter(c["meta"]["h1"] for c in chunks).items():
    print(f"{n:4} | {h1}")

no_title = [w for w in wants if not any(w in c["meta"]["title"] for c in chunks)]
print(f"\n골든 정답 제목 확보: {len(wants) - len(no_title)}/{len(wants)}", no_title)

by_len = sorted(chunks, key=lambda c: len(c["text"]))
print("\n[가장 짧은 8개]")
for c in by_len[:8]:
    print(len(c["text"]), c["meta"]["page"], repr(c["text"][:80]))
print("\n[가장 긴 5개]")
for c in by_len[-5:]:
    print(len(c["text"]), c["meta"]["page"], c["meta"]["title"])

random.seed(0)
print("\n[무작위 5개]")
for c in random.sample(chunks, 5):
    print(c["id"], "p.", c["meta"]["page"])
    print(c["text"][:250], "\n---")

길이 min/중앙/max: 69 517 6707
빈 텍스트: 0
탭 잔존: 0
머리글 잔존: 1
점선 잔존: 0
청크가 닿지 않은 쪽: 3 [2, 3, 4]

[장별 청크 수]
   1 | Legal Information
 105 | Playing the Game
  42 | Character Creation
 340 | Classes
  26 | Character Origins
  19 | Feats
 178 | Equipment
 368 | Spells
 157 | Rules Glossary
  79 | Gameplay Toolbox
 280 | Magic Items
  34 | Monsters
 235 | Monsters A–Z
  94 | Animals

골든 정답 제목 확보: 27/27 []

[가장 짧은 8개]
69 98 'Equipment > Adventuring Gear > Jug (2 CP)\nA Jug holds up to 1 gallon.'
71 97 'Equipment > Adventuring Gear > Flask (2 CP)\nA Flask holds up to 1 pint.'
71 100 'Equipment > Adventuring Gear > Vial (1 GP)\nA Vial holds up to 4 ounces.'
73 176 'Rules Glossary > Rules Definitions\nHere are definitions of various rules.'
76 99 'Equipment > Adventuring Gear > Quiver (1 GP)\nA Quiver holds up to 20 Arrows.'
81 99 'Equipment > Adventuring Gear > Pot, Iron (2 GP)\nAn Iron Pot holds up to 1 gallon'
83 199 'Gameplay Toolbox > Traps > Example Traps\nTraps are presented in alphabetical ord

In [ ]:
# Ex13. jsonl 저장 + 다시 읽어 비교
with open(OUT_PATH, "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")   # 한 줄 = 청크 하나

back = [json.loads(line) for line in open(OUT_PATH, encoding="utf-8")]
print("저장:", len(chunks), "| 다시 읽음:", len(back), "| 동일:", back == chunks)

저장: 1958 | 다시 읽음: 1958 | 동일: True


In [ ]:
# Ex14. 저장한 청크 읽기 -> 임베딩 -> .npy 캐시
from tqdm.auto import tqdm

# 1) jsonl에서 다시 읽기 (이후부터는 PDF 셀을 안 돌려도 여기서 시작 가능)
chunks = [json.loads(line) for line in open(OUT_PATH, encoding="utf-8")]
chunks = [c for c in chunks if c["meta"]["type"] == "rule"]     # 약관(meta)은 검색 대상 아님
print("검색 대상 청크:", len(chunks))

# 2) 여러 분장을 한 번에 임베딩 (Ollama / api / embed, input에 리스트)
def embed_batch(texts, model=EMBED_MODEL):
    res = requests.post(
        f"{OLLAMA_URL}/api/embed",
        json={"model": model, "input": texts, "options": {"num_ctx": 8192}},
        timeout=300,
    )
    res.raise_for_status()
    return res.json()["embeddings"]

print(len(embed_batch(["Opportunity Attacks"])[0]))    # 1024

# 3) 전체 청크 임베딩 + 저장
EMB_PATH = f"{DATA}/rules_new_emb.npy"
BATCH = 32
vecs = []
for i in tqdm(range(0, len(chunks), BATCH)):
    vecs += embed_batch([c["text"] for c in chunks[i:i + BATCH]])

emb = np.array(vecs, dtype="float32")
np.save(EMB_PATH, emb)
print(emb.shape)       # (1957, 1024)

검색 대상 청크: 1957
1024


  0%|          | 0/62 [00:00<?, ?it/s]

(1957, 1024)


In [ ]:
# Ex15. FAISS 인덱스 만들기 (규칙용)
import faiss

vecs = emb.copy()                               # Ex14에서 만든 (1957, 1024) 벡터
faiss.normalize_L2(vecs)                        # 길이를 1로 맞춤 → 내적 = 코사인 유사도
index = faiss.IndexFlatIP(vecs.shape[1])        # 1024차원 벡터를 내적 기준으로 정확히 비교하는 인덱스
index.add(vecs)                                 # i번째로 넣은 벡터 = chunks[i]
print("인덱스에 들어간 벡터 수:", index.ntotal)     # 1957

인덱스에 들어간 벡터 수: 1957


In [ ]:
# Ex16. 검색 확인
golden = json.load(open(f"{DATA}/golden.json", encoding="utf-8"))
ALIAS = {"Grappl": {"grappled", "grappling"}}       # 줄여 적은 정답 보정

def norm(title):
    title = re.sub(r"\s*\[[^\]]*\]$", "", title)      # 이름 끝의 [..] 표시 제거
    title = re.sub(r"\s*\([^)]*\)$", "", title)       # 이름 끝의 (..) 표시 제거
    return title.strip().lower()

def is_hit(title, want):                                # # 이름이 정확히 같을 때만 정답
    return norm(title) == norm(want) or norm(title) in ALIAS.get(want, ())

def wants_of(g):
    return [g["want"]] if isinstance(g["want"], str) else g ["want"]

K = 3
q_vecs = np.array(embed_batch([g["q"] for g in golden]), dtype="float32")
faiss.normalize_L2(q_vecs)
scores, ids = index.search(q_vecs, K)       # ids[i] = 질문 i와 가까운 조각 번호 3개

top1 = top3 = 0
for g, row in zip(golden, ids):
    titles = [chunks[i]["meta"]["title"] for i in row]
    hits = [any(is_hit(t, w) for w in wants_of(g) for t in titles)] # 3개 각각 정답인지
    top1 += hits[0]                                                 # True는 1로 더해짐
    top3 += any(hits)
    if not hits[0]:                                         # 1등이 정답이 아닌 질문만 출력
        mark = "3개 안에 있음" if any(hits) else "못 찾음"
        print(f"[{mark}] {g['q']})\n       정답: {g['want']} | 검색: {titles}")

print(f"1등이 정답: {top1}/{len(golden)} | 3개 안에 정답: {top3}/{len(golden)}")

[못 찾음] How much movement does it cost to stand up after being knocked down?)
       정답: Prone | 검색: ['Climbing', 'Mounting and Dismounting', 'Falling Off']
[못 찾음] What can a blinded creature do?)
       정답: Blinded | 검색: ['Blindsight', 'Blindness/Deafness', 'Lesser Restoration']
[못 찾음] What roll do you make to hide from enemies?)
       정답: Hide | 검색: ['Repelling Blast', 'Level 7: Defensive Tactics', 'Arrow-Catching Shield']
1등이 정답: 22/25 | 3개 안에 정답: 22/25


In [ ]:
# Ex17. 못 찾은 3개: 정답이 상위 30개 중 몇 등에 있나
miss = [g for g in golden if g["want"] in ("Prone", "Blinded", "Hide")]
q = np.array(embed_batch([g["q"] for g in miss]), dtype="float32")
faiss.normalize_L2(q)
scores, ids = index.search(q, 30)

for g, srow, row in zip(miss, scores, ids):
    rank = next((r + 1 for r, i in enumerate(row)
                 if any(is_hit(chunks[i]["meta"]["title"], w) for w in wants_of(g))), None)
    print(g["q"], "->", "정답 등수:", rank)
    if rank:
        print("   1등 점수:", round(float(srow[0]), 3), "| 정답 점수:", round(float(srow[rank - 1]), 3))
        print("   정답 조각:", chunks[row[rank - 1]]["text"][:150].replace("\n", " | "))

How much movement does it cost to stand up after being knocked down? -> 정답 등수: None
What can a blinded creature do? -> 정답 등수: 5
   1등 점수: 0.579 | 정답 점수: 0.557
   정답 조각: Rules Glossary > Rules Definitions > Blinded [Condition] | While you have the Blinded condition, you experience the following effects. Can’t See. You ca
What roll do you make to hide from enemies? -> 정답 등수: None


In [ ]:
# Ex18. 정답 조각의 실제 내용과 질문과의 점수
for g, qv, srow in zip(miss, q, scores):
    if g["want"] == "Blinded":
        continue
    print("\n질문:", g["q"], "| 1등 점수:", round(float(srow[0]), 3))
    for i, c in enumerate(chunks):
        if any(is_hit(c["meta"]["title"], w) for w in wants_of(g)):
            print(f"  정답 후보 [{i}] p.{c['meta']['page']} 길이 {len(c['text'])} | 점수 {float(vecs[i] @ qv):.3f}")
            print("   ", c["text"][:300].replace("\n", " | "))


질문: How much movement does it cost to stand up after being knocked down? | 1등 점수: 0.553
  정답 후보 [1193] p.186 길이 541 | 점수 0.467
    Rules Glossary > Rules Definitions > Prone [Condition] | While you have the Prone condition, you experience the following effects. Restricted Movement. Your only movement options are to crawl or to spend an amount of movement equal to half your Speed (round down) to right yourself and thereby end the 

질문: What roll do you make to hide from enemies? | 1등 점수: 0.558
  정답 후보 [1157] p.183 길이 735 | 점수 0.482
    Rules Glossary > Rules Definitions > Hide [Action] | With the Hide action, you try to hide yourself. To do so, you must succeed on a DC 15 Dexterity (Stealth) check while you’re Heavily Obscured or behind Three-Quarters Cover or Total Cover, and you must be out of any enemy’s line of sight; if you can


In [ ]:
# Ex19. 검색 → 근거 만들기 → 프롬프트 채우기 → 답변 (+ 중국어 섞이면 다시 시도)
def search_rules(question, k=5):
    q = np.array(embed_batch([question]), dtype="float32")
    faiss.normalize_L2(q)
    scores, ids = index.search(q, k)
    return [chunks[i] for i in ids[0]]                 # 비슷한 룰북 조각 k개

SYSTEM = ("You are a D&D rules assistant. Always answer in Korean (Hangul) only. "
          "Never use Chinese characters or Chinese words.")

def ask_llm(prompt, model=LLM_MODEL, temperature=0.1):
    res = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={"model": model,
              "messages": [{"role": "system", "content": SYSTEM},     # 수업 [3]A Ex13: system 역할
                           {"role": "user", "content": prompt}],
              "stream": False,
              "options": {"num_ctx": 8192, "temperature": temperature}},
        timeout=300,
    )
    res.raise_for_status()
    return res.json()["message"]["content"]

TERMS = ("Blinded=실명, Advantage=이점, Disadvantage=불리점, Ability Check=능력 판정, "
         "Attack Roll=공격 굴림, Saving Throw=내성 굴림")      # 용어 통일 (필요하면 계속 추가)

RAG_TEMPLATE = """당신은 D&D 규칙 안내자입니다. 아래 규칙을 반드시 지키세요.
1. [근거]에 적힌 내용만 사용하세요. 근거에 없으면 "근거에서 찾지 못했어요"라고 답하세요.
2. 질문과 관련 없는 근거는 무시하세요.
3. 한국어로만 답하세요. 규칙 용어는 영어 원문을 괄호로 함께 적으세요. 예: 실명(Blinded)
4. 용어는 아래 표의 한국어만 쓰고, 음역(예: 블라인드)은 쓰지 마세요: {terms}
5. 근거에 있는 효과를 빠짐없이 쓰되 다섯 문장 이내로 쓰세요. 이점과 불리점은 누구의 굴림인지(자신의 공격인지, 자신을 향한 공격인지) 구분해서 쓰세요.

### 근거
{context}

### 질문
{question}

### 답변 (한국어로만)"""

def has_chinese(text):                                 # 중국어 글자가 섞였는지 코드로 확인
    return bool(re.search(r"[\u4e00-\u9fff]", text))

def rag_answer(question, k=5, tries=3):
    found = search_rules(question, k)
    lines = []
    for c in found:                                    # "[제목 p.쪽] 본문"
        lines.append(f"[{c['meta']['title']} p.{c['meta']['page']}] {c['text'][:2000]}")
    context = "\n\n".join(lines)
    prompt = RAG_TEMPLATE.format(terms=TERMS, context=context, question=question)

    for n in range(1, tries + 1):                      # 중국어가 섞이면 temperature를 올려 다시 생성
        answer = ask_llm(prompt, temperature=0.1 + 0.3 * (n - 1))
        if not has_chinese(answer):
            break
    sources = [f"{c['meta']['title']} (p.{c['meta']['page']})" for c in found]
    return {"answer": answer, "sources": sources, "tries": n, "ok": not has_chinese(answer)}

for g in golden[:6]:                                   # 골든 질문 앞 6개
    r = rag_answer(g["q"])
    found = any(is_hit(s.rsplit(" (p.", 1)[0], w) for s in r["sources"] for w in wants_of(g))
    print("질문:", g["q"])
    print("답변:", r["answer"])
    print("시도:", r["tries"], "| 정답 항목이 출처에 있음:", found, "\n")

질문: How much does cover improve a target's Armor Class?
답변: Half Cover에서는 +2bonus를, Three-Quarters Cover에서는 +5bonus를 제공합니다.
시도: 1 | 정답 항목이 출처에 있음: True 

질문: When does moving away from an enemy provoke an attack?
답변: 적에게 떨어져 움직일 때 적이 이점으로 공격할 수 있습니다. 적이 보고 있으며, 자신이 적의 닿는 거리에서 벗어나려고 할 때 이점으로 공격을 시도합니다.
시도: 1 | 정답 항목이 출처에 있음: True 

질문: How much movement does it cost to stand up after being knocked down?
답변: 근거에서 찾지 못했어요.
시도: 1 | 정답 항목이 출처에 있음: False 

질문: How far does a torch shed bright light?
답변: 20피트
시도: 1 | 정답 항목이 출처에 있음: True 

질문: What happens if I shoot a bow while an enemy is right next to me?
답변: 이점이 없습니다. 근거에서 찾지 못했어요.
시도: 1 | 정답 항목이 출처에 있음: True 

질문: How is Concentration on a spell broken?
답변: 다른 집중 효과, 손상, 무력화 또는 죽음으로 인해 Concentration이 깨질 수 있습니다.
시도: 1 | 정답 항목이 출처에 있음: True 



In [ ]:
# Ex20. 틀린 답변의 근거 조각 확인 (기회 공격 / 활 쏘기 / 집중)
for idx in (1, 4, 5):
    g = golden[idx]
    print("질문:", g["q"])
    for c in search_rules(g["q"]):
        if any(is_hit(c["meta"]["title"], w) for w in wants_of(g)):
            print(c["text"][:900], "\n")


질문: When does moving away from an enemy provoke an attack?
Playing the Game > Combat > Melee Attacks > Opportunity Attacks
Combatants watch for enemies to drop their guard. If you move heedlessly past your foes, you put yourself in danger by provoking an Opportunity Attack. Avoiding Opportunity Attacks. You can avoid provoking an Opportunity Attack by taking the Disengage action. You also don’t provoke an Opportunity Attack when you Teleport or when you are moved without using your movement, action, Bonus Action, or Reaction. For example, you don’t provoke an Opportunity Attack if an explosion hurls you out of a foe’s reach or if you fall past an enemy. Making an Opportunity Attack. You can make an Opportunity Attack when a creature that you can see leaves your reach. To make the attack, take a Reaction to make one melee attack with a weapon or an Unarmed Strike against that creature. The attack occurs right before it leaves your reach. 

Rules Glossary > Rules Definitions > Opportunit

In [ ]:
# Ex21. 프롬프트 교체: 근거 문장 먼저 복사 → 한국어 설명  (수업 [4] Ex06, Ex11)
TERMS = ("Blinded=실명, Advantage=이점, Disadvantage=불리점, Ability Check=능력 판정, "
         "Attack Roll=공격 굴림, Saving Throw=내성 굴림, Opportunity Attack=기회 공격, "
         "Reaction=반응, Disengage=이탈, Concentration=집중, Incapacitated=행동 불능")

RAG_TEMPLATE = """당신은 D&D 규칙 안내자입니다. 아래 규칙과 형식을 반드시 지키세요.

규칙
1. [근거]에 적힌 내용만 사용하세요. 근거에 없으면 "근거에서 찾지 못했어요"라고만 답하세요.
2. 질문과 관련 없는 근거는 무시하세요.
3. 설명은 한국어로만 쓰고, 규칙 용어는 영어 원문을 괄호로 함께 적으세요. 예: 실명(Blinded)
4. 용어는 아래 표의 한국어만 쓰고, 음역은 쓰지 마세요: {terms}
5. 근거의 내용을 빼거나 뒤집지 마세요. 특히 Advantage와 Disadvantage를 바꾸지 마세요.

형식
[근거 문장] 질문에 답하는 핵심 문장을 근거에서 영어 그대로 한두 문장 복사
[설명] 위 문장을 바탕으로 한 한국어 설명 (다섯 문장 이내)

### 근거
{context}

### 질문
{question}

### 답변"""

# 답에 반드시 들어가야 할 말 (골든 질문 번호: 단어). 수업 [4] Ex04처럼 코드로 확인
MUST = {0: ["+2", "+5"], 1: ["Reaction"], 3: ["20"], 4: ["Disadvantage"], 5: ["Constitution"]}

passed = 0
for idx, words in MUST.items():
    r = rag_answer(golden[idx]["q"])
    ok = all(w in r["answer"] for w in words)
    passed += ok
    print("통과" if ok else "실패", "|", golden[idx]["q"])
    print(r["answer"], "\n")
print(f"통과 {passed}/{len(MUST)}")

통과 | How much does cover improve a target's Armor Class?
[Cover p.15] Playing the Game > Combat > Cover
Half Cover +2 bonus to AC and Dexterity saving throws
Three-Quarters Cover +5 bonus to AC and Dexterity saving throws
[Cover p.179] Rules Glossary > Rules Definitions > Cover
Half Cover (+2 bonus to AC and Dexterity saving throws), Three-Quarters Cover (+5 bonus to AC and Dexterity saving throws)

[설명] Half Cover는 2의 AC 보너스를 제공하고, Three-Quarters Cover는 5의 AC 보너스를 제공합니다. 이 보너스는 던전 모험에서 공격을 피하는 데 도움이 됩니다. 

실패 | When does moving away from an enemy provoke an attack?
[Opportunity Attacks p.15] Combatants watch for enemies to drop their guard. If you move heedlessly past your foes, you put yourself in danger by provoking an Opportunity Attack. Avoiding Opportunity Attacks. You can avoid provoking an Opportunity Attack by taking the Disengage action.
[설명] 적과 거리를 벌리면서 무방비 상태로 움직이면 기회 공격을 받을 위험성이 있습니다. 이 상황을 피하려면 이탈(Disengage) 행동을 취할 수 있습니다. 

통과 | How far does a torch shed bright light?
[ 

In [ ]:
# Ex22. 프롬프트 교체: 예시 1개 추가 (수업 [4] Ex06 In-Context Learning)
RAG_TEMPLATE = """당신은 D&D 규칙 안내자입니다. 아래 규칙을 반드시 지키세요.
1. [근거]에 적힌 내용만 사용하세요. 근거에 없으면 "근거에서 찾지 못했어요"라고만 답하세요.
2. 질문과 관련 없는 근거는 무시하세요.
3. 용어는 이 표의 한국어를 쓰고 영어 원문을 괄호로 함께 적으세요: {terms}
4. 근거의 조건, 숫자, 이점/불리점을 빼거나 바꾸지 마세요.
5. 답변은 "원문:"과 "설명:" 두 줄로 쓰세요. 원문은 질문에 직접 답하는 문장을 근거에서 영어 그대로 복사하고, 설명은 한국어로 쓰세요.

### 예시
질문: What does the Dash action do?
근거: [Dash [Action] p.181] When you take the Dash action, you gain extra movement for the current turn. The increase equals your Speed after applying any modifiers.
답변:
원문: When you take the Dash action, you gain extra movement for the current turn. The increase equals your Speed after applying any modifiers.
설명: 질주(Dash) 행동을 하면 이번 턴에 속도(Speed)만큼 이동 거리가 늘어납니다.

### 근거
{context}

### 질문
{question}

### 답변"""

passed = 0
for idx, words in MUST.items():
    r = rag_answer(golden[idx]["q"])
    ok = all(w in r["answer"] for w in words)
    passed += ok
    print("통과" if ok else "실패", "|", golden[idx]["q"])
    print(r["answer"], "\n")
print(f"통과 {passed}/{len(MUST)}")

실패 | How much does cover improve a target's Armor Class?
원문: A target can benefit from cover only when an attack or other effect originates on the opposite side of the cover. If a target is behind multiple sources of cover, only the most protective degree of cover applies; the degrees aren’t added together.
설명: 대상은 공격이나 다른 효과가 커버의 반대편에서 발생할 때만 커버의 이점을 받을 수 있습니다. 대상이 여러 개의 커버 뒤에 있을 때, 가장 보호력이 높은 커버의 이점을만 받으며, 이점들은 더해지지 않습니다. 

실패 | When does moving away from an enemy provoke an attack?
원문: You can make an Opportunity Attack when a creature that you can see leaves your reach.
설명: 적과의 거리를 벗어나려고 움직일 때, 즉 적을 볼 수 있는 상태에서 거리 범위를 벗어나려고 움직일 때, 기회 공격을 받을 수 있습니다. 

통과 | How far does a torch shed bright light?
원문: A Torch burns for 1 hour, casting Bright Light in a 20-foot radius and Dim Light for an additional 20 feet.
설명: 토르치는 20피트의 밝은 빛을 발산하며, 추가로 20피트의 약한 빛을 발산합니다. 

통과 | What happens if I shoot a bow while an enemy is right next to me?
원문: When you make a ranged attack roll with a weapon, a s